# Assignment 1 - Evaluation

### Intro
This assignment is a hands-on dive into LLM evaluation: from understanding the business use-case, through defining evaluation criteria, to performing evaluation using two methods - human evaluation and Judge Model (aka LLM-as-a-judge). Along the way you'll explore the strengths and trade-offs of each method, and grapple with the inherent challenge of evaluating outputs that have no single "correct" answer.

### The business use case: generate product descriptions
The provided dataset contains e-commerce products with the following features: name, structured attributes, material, and warranty. Later you’ll be asked to use a language model for crafting a persuasive, 50–90 word description of each product based on these features.

### The evaluation criteria:
| Criterion | Description | Rating |
| :--- | :--- | :--- |
| **Fluency** | Natural, easy‑to‑read sentences | good / ok / bad |
| **Grammar** | Correct spelling & punctuation | good / ok / bad |
| **Tone** | Matches friendly, credible sales voice | good / ok / bad |
| **Length** | 50‑90 words | good / ok / bad |
| **Grounding** | Sticks to information provided | good / ok / bad |
| **Latency** (avg. time per call) | Time to first byte / full response | good / ok / bad |
| **Cost** (avg. price per call) | Relative inference or API cost per 1K tokens | good / ok / bad |

## Task 1 - define your rubric
Before generating or evaluating anything, you need a clear, repeatable scoring framework. Define it now so that every evaluation decision - whether made by you or by a judge model later - follows the same rules.
1. Criterion definitions - For each criterion in the table above, write explicit definitions for good, ok, and bad. The goal is to minimize subjectivity: another person reading your rubric should reach the same verdict you would. Example for Length: good = 50–90 words, ok = 40–49 or 91–110 words, bad = outside that range.
2. Pass / fail definition:
a. Cumulative pass bar - Define the minimum combination of ratings a description must achieve to pass. For example: "at least three good ratings and zero bad ratings."
b. Go / no-go rules - Define any single criterion that triggers automatic failure regardless of other scores. For example: "if Grounding is not good, the description is rejected."


**Criterion definitions**
1. Grounding: “Sticks to information provided”.
    Grounding evaluates whether the factual claims in the description are supported by the provided product attributes. It does not evaluate writing quality (Fluency), voice (Tone), or mechanical correctness (Grammar).

    * **Good**: Every factual claim in the description is directly supported by the provided product structured attributes. Generic marketing language (“great choice”, “you’ll love it”) is acceptable since it makes no specific factual claim. The following category-based inferences are allowed:
    - Physical/sensory properties from materials (steel → durable, cotton → soft, leather → premium-feeling)
    - Functional capabilities from structured attributes (Bluetooth 5.0 → wireless connectivity, 12-speed gear → versatile shifting)

    * **Ok**: The description is mostly faithful, but contains at least one instance of unsupported-but-plausible opinion or suggestion: a subjective claim that isn’t in the input but doesn’t contradict it and cannot be verified as true or false.
    Examples of claims that trigger an Ok rating (downgrade from Good): “perfect for hiking”, “makes a great gift”, “eco-friendly choice”, “ideal for commuters”, “great for travel”, or any other lifestyle / values / contextual use-case claim that requires assumptions beyond the product’s category and attributes.
    * **Bad**: The description contradicts provided features (e.g., stating “titanium” instead of steel) or contains one or more fabricated specific/technical claims (e.g., “waterproof to 30 meters”, “available in 5 colors”, “FDA-approved”).

2. Length (50-90 words):
    * **Good**: 50-90 words
    * **Ok**: 40-49 or 91-110 words
    * **Bad**: fewer than 40 or more than 110 words

3. Latency (avg. time per call):
    * **Good**: ≤ 5 seconds
    * **Ok**: 5-8 seconds
    * **Bad**: > 8 seconds

4. Cost (avg. price per call):
    * **Good**: ≤ $0.00006 per call
    * **Ok**: $0.00006-$0.0001 per call
    * **Bad**: > $0.0001 per call

5. Grammar (correct spelling & punctuation):
    * **Good**: No spelling, punctuation, or grammatical errors.
    * **Ok**: Contains 1-2 minor errors that don't hinder readability (e.g., a missing comma, a minor typo).
    * **Bad**: Contains 3+ errors, OR any error that changes meaning or makes a sentence hard to understand.

6. Fluency: "Natural, easy-to-read sentences"
   Fluency evaluates whether the description reads smoothly as
   coherent, human-sounding prose. It does NOT evaluate whether
   the voice is appropriate for e-commerce (Tone), whether the
   facts are accurate (Grounding), or whether spelling and
   punctuation are correct (Grammar).
    - **Flow/cohesion** - the description reads as a unified paragraph, not a list of disconnected facts. Observable markers: sentences reference or build on each other, related features are grouped together, there's some logical progression (e.g., what it is -> what it's made of -> what that means for you).
    - **Naturalness** - the description sounds like something a human copywriter would write. Observable markers: varied sentence structure (not every sentence follows the same pattern), no awkward or stilted phrasing, doesn't read like bullet points reformatted into sentences.

    * **Good**: Both components satisfied - cohesive flow and natural-sounding.
    * **Ok**: One component breaks down - e.g., natural phrasing but sentences feel disconnected, or well-structured but with some robotic/repetitive patterns.
    * **Bad**: Both components fail - reads like a list of disjointed facts in awkward or repetitive phrasing.

7. Tone - "Friendly, credible sales voice"
   Tone evaluates whether the description uses the right *voice*
   for e-commerce product copy. It does NOT evaluate whether the text
   flows well (Fluency) or is mechanically correct (Grammar).

   Two components:
   - **Friendliness**: warm, approachable, benefit-oriented.
     Speaks *to* the customer about what the product does for them,
     not *at* them with raw specs.
   - **Credibility**: confident but measured. No unsupported
     superlatives ("the BEST ever"), no high-pressure urgency
     ("buy NOW before it's gone!"), no stacked vague intensifiers
     ("truly absolutely incredible").

   * **Good**: Both satisfied - warm and inviting without
     overselling. Uses benefit-oriented language, feels professional
     and approachable.
   * **Ok**: Leans too far in one direction - either too
     dry/clinical (spec-sheet voice, lists features without
     connecting to benefits) OR mildly over-enthusiastic (some
     unnecessary superlatives, slightly salesy) but still broadly
     appropriate for e-commerce.
   * **Bad**: Clearly wrong voice - either reads like a technical
     manual with zero sales appeal, OR is aggressively
     pushy/hyperbolic (infomercial-style pressure, excessive
     exclamation marks, high-pressure urgency, stacked superlatives).

Pass / fail definition

1.   Cumulative pass bar: At least 3 good ratings and 0 bad ratings.
2.   List Go / no-go rules:
      - If Grounding is not GOOD, the description is rejected.
      - If Grammar is BAD, the description is rejected.
      - If Fluency is BAD, the description is rejected.
      - If Tone is BAD, the description is rejected.
      - If Grammar + Fluency + Tone is OK, the description is rejected.



## Task 2 - generate the description for every product

### Prompt:

```
Using the rubric you defined, now write a system prompt and use it to generate a description for every product in the dataset. Recall that the prompt should instruct the model to produce a persuasive 50–90 word product description based on the provided information (name, structured attributes, material, warranty). Apply the prompting guidelines covered in class.
```

### Model:

Choose one of the following models from Nebius Token Factory and use it for generating the descriptions:
1. Gemma-2-9b-it
2. Meta-Llama-3.1-8B-Instruct
Structured Output
For each API call, collect the following fields into a dictionary:
- generated_description
- latency_ms - end‑to‑end generation time in milliseconds
- input_tokens - number of tokens sent to the model (including the system prompt)
- output_tokens - number of tokens received from the model

### Storage:

Collect all results into a DataFrame. Add blank columns for each rubric criterion and a blank final_score column (pass / fail). Save the DataFrame as an Excel file named assignment_01.xlsx.

### Note:
the models above are relatively small, and you’ll probably get some bad results. That’s actually good since it will give you interesting results to evaluate and improve later on. You won’t lose points for bad descriptions as long as all the rest is correct.

In [60]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

True

In [ ]:
system_prompt = """
 You are an e-commerce product copywriter. Your task is to write a single
 persuasive product description based ONLY on the provided product 
 information.
 
 STRICT RULES: 
 
 1. LENGTH: Write exactly 50 to 90 words. Count carefully. 
 
 2. GROUNDING - this is the most critical rule:
- ONLY use facts explicitly stated in the provided product name,
 attributes, material, and warranty. 
- You MAY infer physical properties from materials (e.g., steel =
 durable, cotton = soft, aluminum = lightweight).
- You MAY infer functional capabilities from attributes (e.g., Bluetooth
 5.2 = wireless connectivity, 30 hr battery = long listening sessions).
- You MUST NOT invent features, specs, or claims not in the input (e.g.,
 do not add "waterproof", color options, or certifications unless provided). 
- You MUST NOT add lifestyle or use-case claims not in the input (e.g.,
 do not say "perfect for hiking", "eco-friendly choice", or "great gift
 idea"). 
- Generic marketing phrases with no factual claim are fine (e.g., "you\'ll
love it", "a great choice"). 
 
 3. TONE - 4 friendly and credible:
- Write in a warm, benefit-oriented voice. Speak TO the customer about
 what the product does for them. 
- Stay confident but measured. No superlatives ("the BEST ever"), no
 pressure tactics ("buy NOW!"), no stacked intensifiers ("truly absolutely 
 incredible"). 
- Do not write like a dry spec sheet. Connect features to benefits.
 
 4. FLUENCY- natural, cohesive prose:
- Write a unified paragraph, not bullet points or a list of disconnected 
 facts.
- Vary sentence structure. Group related features together. Build a
 logical flow (what it is -> what it\'s made of -> what that means for the
 customer).
- It should sound like a human copywriter, not a reformatted spec list.
 
 5. GRAMMAR: Use correct spelling, punctuation, and grammar throughout. Zero 
 errors.

 OUTPUT: Return ONLY the product description. No titles, labels, headers, or 
 extra commentary.
"""

In [44]:

# Initiating LLM client
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",  # Nebius endpoint
    api_key=os.environ.get("NEBIUS_API_KEY")                                          
)

In [45]:
# Read CSV into dataframe
df = pd.read_csv("products_dataset.csv")

In [46]:
results = []                                                                

for _, row in df.iterrows():
    user_msg = f"""Product: {row['product_name']}
    Attributes: {row['Product_attribute_list']}                                 
    Material: {row['material']}
    Warranty: {row['warranty']}"""                                              
                
    start = time.time()                                                     
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[                                                          
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}                           
        ],      
        temperature=0.4
    )                                                                       
    latency_ms = (time.time() - start) * 1000
                                                                            
    results.append({
        "product_name": row["product_name"],
        "generated_description": response.choices[0].message.content,       
        "latency_ms": round(latency_ms),
        "input_tokens": response.usage.prompt_tokens,                       
        "output_tokens": response.usage.completion_tokens,
    })                                                                      


In [47]:
# Calculate average latency
avg_latency_ms = sum(r["latency_ms"] for r in results) / len(results)       
                                                                            
# Calculate average cost per call                                           
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens      
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens     
                                                                            
costs = []                                                                  
for r in results:                                                           
    cost = (r["input_tokens"] * input_price_per_token) + (r["output_tokens"]
* output_price_per_token)                                                  
    costs.append(cost)
                                                                            
avg_cost = sum(costs) / len(costs)

print(f"Average latency: {avg_latency_ms:.0f} ms")                          
print(f"Average cost per call: ${avg_cost:.6f}")

Average latency: 2084 ms
Average cost per call: $0.000033


In [48]:
results

[{'product_name': 'Apple iPhone 15 Pro',
  'generated_description': 'Experience seamless performance with the Apple iPhone 15 Pro, powered by the A17 Pro chip. Enjoy a responsive and immersive display with 120Hz ProMotion technology, perfect for gaming and video streaming. The compact design fits comfortably in your hand, while the durable titanium frame and Ceramic Shield glass provide long-lasting protection. With USB-C fast charging, you can quickly top up your battery and get back to using your device.',
  'latency_ms': 1885,
  'input_tokens': 847,
  'output_tokens': 85},
 {'product_name': 'Samsung Galaxy S24 Ultra',
  'generated_description': "Capture stunning, high-resolution photos with the 200 MP camera, perfect for capturing life's details. The S-Pen support allows for precise note-taking and creative expression. Enjoy seamless visuals with the 120 Hz AMOLED display. Built with a durable Armor Aluminum frame and Gorilla Glass Victus, this phone withstands daily wear. With a 1-

In [ ]:
# Store results
results_df = pd.DataFrame(results)

for criterion in ["Fluency", "Grammar", "Tone", "Length", "Grounding",      
"Latency", "Cost"]:                                                         
    results_df[criterion] = ""
results_df["final_score"] = ""                                              
                
results_df.to_excel("assignment_01_task2.xlsx", index=False)

## Task 3 - manual (human) evaluation
Open the assignment_01 sheet you created in task 2: 
1. Cost calculation - Add a cost column. For each row, convert input_tokens and output_tokens into cost in USD using the pricing for your chosen model (note: input and output tokens are priced differently).
2. Rate each criterion - Choose 10-15 products and for each of them rate each
criterion as good / ok / bad according to the rubric you defined in Task 1.
3. Final score - Apply your cumulative pass bar and go/no-go rules from Task 1 to set
the final_score for each row.
4. Baseline analysis - Review your scores across all products. Which criteria
performed best? Which performed worst? Use this analysis to guide your
improvement strategy in the next task.

In [ ]:
# Cost calculation
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens

df_eval = pd.read_excel("assignment_01_task2.xlsx")

df_eval["Cost"] = (
    df_eval["input_tokens"] * input_price_per_token
    + df_eval["output_tokens"] * output_price_per_token
)

df_eval.to_excel("assignment_01_task3.xlsx", index=False)
df_eval[["product_name", "input_tokens", "output_tokens", "Cost"]]

In [ ]:
# 3.2 - Display first 15 products side-by-side with their generated descriptions
products = pd.read_csv("products_dataset.csv")
generated = pd.read_excel("assignment_01_task3.xlsx")

comparison = products.head(15).merge(
    generated[["product_name", "generated_description", "latency_ms", "Cost"]],
    on="product_name"
)

comparison["word_count"] = comparison["generated_description"].str.split().str.len()

# Display full text without truncation
with pd.option_context("display.max_colwidth", None):
    display(comparison)

comparison.to_excel("assignment_01_task3.xlsx", index=False)

## Task 4 - improvement cycle
Now that you’ve established a baseline score, iterate to achieve better results.
Ideas to explore (you don’t have to try all of them!)
- Prompt engineering – rewrite the system prompt, add/change few‐shot examples,
or enforce stricter constraints.
- Model choice – try a different model from the token factory - bigger one (e.g., ~30B),
or different architecture..
- Decoding parameters - adjust temperature, top_p, top_k, or
max_new_tokens to balance creativity vs. factuality.
- Post‐processing – run grammar‐checking or length trimming after generation.
- Ensembling – combine outputs from two models.

For each experiment, document:
1. What you changed
2. Why you expected it to help
3. New evaluation scores (re-evaluate using your Task 1 rubric)

In [ ]:
# Baseline Analysis Overall: 4 pass / 6 fail (40% pass rate) 
# Biggest pain point: 
#  - Grounding:   4 good, 6 ok
#  - Tone:        4 good, 6 ok

# first experiment - improve prompt
# the first thing that comes to mind is to impove the instructions that the model follows
# this experiment is improving only the system prompt
# We expected it to help because the baseline failures were mainly caused by                                                        
# the model adding unsupported claims (grounding) and over-promotional language                                         
# (tone). A more specified system prompt with strict grounding rules,                                                   
# examples of grounded vs. not-grounded claims, and clear boundaries on lifestyle/                                                     
# audience language should directly reduce these two failure modes.

import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

EXPERIMENT_FILE="experiment1.xlsx"

system_prompt = """
   You are an e-commerce product copywriter. Your task is to write a single
   persuasive product description based ONLY on the provided product
   information.

   STRICT RULES:

   1. LENGTH: Write exactly 50 to 90 words. Count carefully.

   2. GROUNDING - this is the most critical rule:
      - ONLY use facts explicitly stated in the provided product name,
   attributes, material, and warranty.
      - You MAY infer physical properties from materials (e.g., steel =
   durable, cotton = soft, aluminum = lightweight).
      - You MAY infer functional capabilities from attributes (e.g., Bluetooth
   5.2 = wireless connectivity, 30 hr battery = long listening sessions).
      - You MUST NOT describe inherent use-case implications that follow directly
   from the product category combined with its stated attributes (e.g.,
   4K/60fps drone = "capture stunning aerial footage", 200 MP camera phone =
   "sharp, detailed photos").
      - You MUST NOT invent features, specs, or claims not in the input (e.g.,
   do not add "waterproof", color options, or certifications unless provided).
      - You MUST NOT add lifestyle, contextual, or target-audience claims
  that go beyond what the product category and its attributes imply (e.g.,
  do not say "perfect for hiking", "eco-friendly choice", "great gift idea",
  "ideal for your daily commute", "designed for professional photographers",
  or "perfect for graphic designers").
      - Generic marketing phrases with no factual claim are fine (e.g.,
   "you'll love it", "a great choice").

      GROUNDING EXAMPLES:
      ✓ GROUNDED (inherent implication from specs):
        Input: "30 hr battery, synthetic leather earcups"
        Output: "Enjoy long listening sessions with 30 hours of battery life"

      ✗ NOT GROUNDED (contextual use-case claim):
        Input: "30 hr battery, synthetic leather earcups"
        Output: "Perfect for long flights or your daily commute"

      ✗ NOT GROUNDED (target-audience claim):
         Input: "100% sRGB, Calman Verified, metal stand"
         Output: "Perfect for graphic designers and professionals"
         → Specifies who should buy the product - not stated in the attributes.
         Instead, describe the capability: "delivers consistent, calibrated color
         accuracy"

      The first describes what the product enables based on its specs. The
      second assumes where and how the customer will use it.

   3. TONE - friendly and credible:
      - Write in a warm, benefit-oriented voice. Speak TO the customer about
   what the product does for them.
      - Stay confident but measured. No superlatives ("the BEST ever"), no
   pressure tactics ("buy NOW!"), no stacked intensifiers ("truly absolutely
   incredible").
      - Do not write like a dry spec sheet. Connect features to benefits.

   4. FLUENCY - natural, cohesive prose:
      - Write a unified paragraph, not bullet points or a list of disconnected
   facts.
      - Vary sentence structure. Group related features together. Build a
   logical flow (what it is → what it's made of → what that means for the
   customer).
      - It should sound like a human copywriter, not a reformatted spec list.

   5. GRAMMAR: Use correct spelling, punctuation, and grammar throughout. Zero
   errors.

   GENERIC RULES:
   1. Be diverse with the vocabulary you use.
   2. Do not use your knowledge of the product - only use the attributes provided.

   OUTPUT: Return ONLY the product description. No titles, labels, headers, or
   extra commentary.
"""

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/", 
    api_key=os.environ.get("NEBIUS_API_KEY")                                          
)

# Read CSV into dataframe
df = pd.read_csv("products_dataset.csv")
total = len(df)

results = []                                                                

print(f"Starting inference for {total} products...")
for idx, row in df.iterrows():
    user_msg = f"""Product: {row['product_name']}
    Attributes: {row['Product_attribute_list']}                                 
    Material: {row['material']}
    Warranty: {row['warranty']}"""                                              
                
    start = time.time()                                                     
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3.1-8B-Instruct",
        messages=[                                                          
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}                           
        ],      
        temperature=0.4
    )                                                                       
    latency_ms = (time.time() - start) * 1000
                                                                            
    results.append({
        "product_name": row["product_name"],
        "generated_description": response.choices[0].message.content,       
        "latency_ms": round(latency_ms),
        "input_tokens": response.usage.prompt_tokens,                       
        "output_tokens": response.usage.completion_tokens,
    })

    print(f"[{idx + 1}/{total}] {row['product_name']} - {latency_ms:.0f}ms")

# Calculate average latency
avg_latency_ms = sum(r["latency_ms"] for r in results) / len(results)       
                                                                            
# Calculate average cost per call                                           
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens      
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens     
                                                                            
costs = []                                                                  
for r in results:                                                           
    cost = (r["input_tokens"] * input_price_per_token) + (r["output_tokens"]
* output_price_per_token)                                                  
    costs.append(cost)
                                                                            
avg_cost = sum(costs) / len(costs)

print(f"\nDone! Average latency: {avg_latency_ms:.0f} ms | Average cost: ${avg_cost:.6f}")

# Store results
results_df = pd.DataFrame(results)

for criterion in ["Fluency", "Grammar", "Tone", "Length", "Grounding",      
"Latency", "Cost"]:                                                         
    results_df[criterion] = ""
results_df["final_score"] = ""                                              
                
results_df.to_excel(EXPERIMENT_FILE, index=False)
print(f"Saved to experiment1.xlsx")

# Cost calculation
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens

df_eval = pd.read_excel(EXPERIMENT_FILE)

df_eval["Cost"] = (
    df_eval["input_tokens"] * input_price_per_token
    + df_eval["output_tokens"] * output_price_per_token
)


"""
   What we have changed: 
      ADDED: 
         - You MUST NOT describe inherent use-case implications that follow directly
               from the product category combined with its stated attributes (e.g.,
               4K/60fps drone = "capture stunning aerial footage", 200 MP camera phone =
               "sharp, detailed photos").
         - GROUNDING EXAMPLES:
            ✓ GROUNDED (inherent implication from specs):
            Input: "30 hr battery, synthetic leather earcups"
            Output: "Enjoy long listening sessions with 30 hours of battery life"

            ✗ NOT GROUNDED (contextual use-case claim):
            Input: "30 hr battery, synthetic leather earcups"
            Output: "Perfect for long flights or your daily commute"

            ✗ NOT GROUNDED (target-audience claim):
               Input: "100% sRGB, Calman Verified, metal stand"
               Output: "Perfect for graphic designers and professionals"
               → Specifies who should buy the product - not stated in the attributes.
               Instead, describe the capability: "delivers consistent, calibrated color
               accuracy"

            The first describes what the product enables based on its specs. The
            second assumes where and how the customer will use it.
         -    GENERIC RULES:
               1. Be diverse with the vocabulary you use.
               2. Do not use your knowledge of the product - only use the attributes provided.


   first experiment results: 
  ┌──────────────┬───────────────────────┬──────────────────────┐   
  │    Metric    │   Baseline (Task 3)   │     Experiment 1     │
  ├──────────────┼───────────────────────┼──────────────────────┤   
  │ Pass rate    │ 4/10 (40%)            │ 8/10 (80%)           │   
  ├──────────────┼───────────────────────┼──────────────────────┤   
  │ Grounding    │ 4/10                  │ 8/10                 │   
  │ good         │                       │                      │
  ├──────────────┼───────────────────────┼──────────────────────┤   
  │ Latency      │ Mixed (many ok/bad at │ All good             │   
  │              │  ~5800ms)             │ (~1400-2800ms)       │
  ├──────────────┼───────────────────────┼──────────────────────┤   
  │ Cost         │ All good              │ All good             │
  ├──────────────┼───────────────────────┼──────────────────────┤
  │ Grammar      │ All good              │ All good             │
  ├──────────────┼───────────────────────┼──────────────────────┤
  │ Fluency      │ All good              │ 9 good, 1 ok (Galaxy │
  │              │                       │  S24)                │   
  ├──────────────┼───────────────────────┼──────────────────────┤
  │ Tone         │ All good              │ All good             │   
  ├──────────────┼───────────────────────┼──────────────────────┤   
  │ Length       │ All good              │ All good             │
  └──────────────┴───────────────────────┴──────────────────────┘   

   Discoveries: 
   - Just from updating the prompt - Grounding jumped from 40% to 80% good AND Tone jumped to 100% good from 40% (!)
   - Latency also improved to 100% good apparently due to network issue we had in the baseline results
"""

Experiment 2 - top_p=0.05 (replacing temperature)  
                                                    
  1. What I changed: Replaced temperature=0.4 with
  top_p=0.05, keeping the improved prompt from        
  Experiment 1.
  2. Why I expected it to help: top_p limits the token
   sampling pool to the most probable tokens, which   
  should make outputs more deterministic and reduce
  creative embellishments that cause grounding        
  failures.       
  3. Result: 6/10 pass (60%) - Worse than Experiment
  1. The model became too constrained and less fluent.
   
  ---                                                 
  Experiment 3 - temperature=0.3
                                 
  1. What I changed: Set temperature to 0.3 (lower
  than Experiment 1's 0.4), returned top_p to default.
  2. Why I expected it to help: Lower temperature
  makes the model more deterministic, which should    
  reduce the creative/unsupported claims that caused
  grounding failures.                                 
  3. Result: 3/10 pass (30%) - Worst result.
  Counterintuitively, the model became more           
  "confident" and fabricated specific material
  properties.                                         
                  
  ---
  Experiment 4 - temperature=0.35
                                                      
  1. What I changed: Slightly increased temperature to
   0.35, splitting the difference between the failing 
  0.3 and the successful 0.4.
  2. Why I expected it to help: Finding the sweet spot
   between too deterministic (0.3) and too creative   
  (default).
  3. Result: 4/10 pass (40%) - No improvement over    
  baseline.                                           
   
  ---                                                 
  Experiment 5 - Prompt v2 (more detailed grounding 
  rules) + temp=0.4                                   
                   
  1. What I changed: Made the system prompt more      
  detailed with additional negative examples and      
  stricter grounding boundaries, keeping temp=0.4.
  2. Why I expected it to help: More explicit         
  instructions and examples should further reduce     
  grounding violations.
  3. Result: 5/10 pass (50%) - Worse than Experiment  
  1. The more complex prompt overwhelmed the 8B model.
   
  ---                                                 
  Experiment 6 - Prompt v3 (further prompt 
  refinement)                                         
   
  1. What I changed: Further restructured and refined 
  the system prompt.
  2. Why I expected it to help: Clearer organization
  of rules might help the model prioritize them       
  better.
  3. Result: 4/10 pass (40%) - No improvement.        
  Confirmed that more complex prompts degrade         
  performance on the 8B model.
                                                      
  ---             
  Experiment 7 - Qwen3-30B-A3B-Thinking model
                                              
  1. What I changed: Switched from Llama 3.1 8B to
  Qwen3-30B-A3B-Thinking (a larger, reasoning-capable 
  model), keeping the same prompt and temp=0.4.
  2. Why I expected it to help: A larger model with   
  reasoning capabilities should better follow complex 
  grounding instructions.
  3. Result: 4/10 pass (40%) - The thinking model     
  inferred too aggressively, adding plausible-sounding
   but unsupported claims. Also introduced latency
  regression.                                         
                  
  ---
  Experiment 8 - Added ending rule + temp=0.4
                                              
  1. What I changed: Added a rule to the prompt: "End
  with the warranty or a generic call-to-action. Do   
  NOT end with lifestyle, motivational, or fashion
  statements."                                        
  2. Why I expected it to help: Several failures were
  caused by lifestyle claims at the end of            
  descriptions (e.g., "fitness journey," "complements
  any outfit"). Constraining the ending should        
  eliminate these.
  3. Result: 6/10 pass (60%) - Garmin and Bose
  improved (lifestyle endings eliminated), but camera 
  products still failed on grounding.
                                                      
  ---             
  Experiment 9 - Same prompt + temp=0.2
                                        
  1. What I changed: Lowered temperature from 0.4 to
  0.2 for more deterministic output, keeping the      
  ending rule.
  2. Why I expected it to help: Lower temp reduces    
  creative embellishments while the ending rule       
  prevents lifestyle claims.
  3. Result: 7/10 pass (70%) - Surface Pro improved   
  (no more superlatives), but Garmin regressed with   
  "active lifestyle" claim.
                                                      
  ---             
  Experiment 10 - Added macro grounding instructions +
   temp=0.4                                           
           
  1. What I changed: Added generalized grounding
  rules: "Do not describe the output or result of a   
  spec" and "Do not use lifestyle descriptors about
  the customer," with negative examples for camera    
  output, display use-cases, and lifestyle
  descriptors.
  2. Why I expected it to help: These macro rules
  target the remaining failure patterns (camera       
  quality claims, display use-cases, lifestyle
  descriptors) without being product-specific.        
  3. Result: 5/10 pass (50%) - Fixed iPhone (first
  time passing!) but broke Dell XPS and Echo Dot.     
  Longer prompt destabilized other products.
                                                      
  ---             
  Experiment 11 - Macro instructions + temp=0.2
                                               
  1. What I changed: Combined the macro grounding
  instructions with lower temperature (0.2).          
  2. Why I expected it to help: Combining the targeted
   instructions with conservative sampling should give
   the best of both approaches.
  3. Result: 6/10 pass (60%) - iPhone still passing,  
  but Dell XPS regressed with "perfect for working on 
  the go."
                                                      
  ---             
  Experiment 12 - gpt-oss-20b model (first run)
                                                      
  1. What I changed: Switched to gpt-oss-20b (a larger
   20B model), keeping the macro grounding            
  instructions and temp=0.2.
  2. Why I expected it to help: A larger model should 
  follow complex multi-rule prompts more reliably.    
  3. Result: 6/10 pass (60%) - Grounding dramatically
  improved (Surface Pro and Garmin passed grounding   
  for the first time), but the model's high output
  token count (thinking tokens) pushed 3 products over
   the cost threshold (Bad).

  ---
  Experiment 13 - gpt-oss-20b (second run)
                                          
  1. What I changed: Re-ran with gpt-oss-20b (same
  configuration).                                     
  2. Why I expected it to help: Hoped for more stable
  output with the same good grounding.                
  3. Result: 3/10 pass (30%) - High variance between
  runs. New issues appeared: broken sentence fragment 
  (Pixel), pronoun confusion (Surface Pro), fabricated
   keyboard claims (MacBook). Cost remained Bad for   
  multiple products. The thinking model proved too
  inconsistent.

  ---                                                  
  Experiment 14 - Rubric update + new prompt        
                  
  1. What I changed: Updated the grounding rubric with 
  clearer boundaries (allowed material inferences,     
  disallowed lifestyle claims). Rewrote the system
  prompt as "technical e-commerce copywriter" with     
  strict factual tone, added DO/DON'T examples for spec
   translation.
  2. Why I expected it to help: The model
  couldn't follow complex prompts (experiments 5-6     
  showed degradation with longer prompts). Address the remaining  
  grounding failures. The new  
  factual tone should eliminate lifestyle/experiential
  claims.
  3. Result: Grounding: 5 Good, 4 Ok, 1 Bad - 
  iPhone failed with fabricated warranty  
  detail ("coverage for parts and labor"), Bose got Ok
  for unsupported "secure fit" claim. 


In [ ]:
# Baseline Analysis Overall: 8 pass / 2 fail (80% pass rate) 
# Biggest pain point: Grounding 8 good, 2 ok

# Second (succesful) experiment - Switched model from Llama 3.1 8B to google/gemma-2-9b-it-fast, temp=0.4.
# The Llama 8B model couldn't follow complex prompts and the more we changed and refined the instructions
# the worse the results got. At some point we realised we got to the limit of the model and
# we couldn't get better results from him so replacing the model is the next thing to push to 100% success.

import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

EXPERIMENT_FILE="experiment2.xlsx"

system_prompt="""
You are a technical e-commerce copywriter. Your task is to write a single product description paragraph (exactly 50 to 90 words) based ONLY on the provided product information.

**TONE AND STYLE:**
Write premium, objective catalog copy. Your tone must be friendly, credible grounded entirely in the physical and functional reality of the product. Focus on the engineering, materials, and specifications. Do not use emotional appeals, superlatives, or lifestyle.

**STRICT GROUNDING RULES:**

1. ONLY state the features, materials, and specifications provided in the input.
2. Describe functional capabilities directly tied to the hardware (e.g., "A 30-hour battery provides extended use," or "Bluetooth 5.2 offers wireless connectivity").
3. STOP there. You must leave out the user's experience entirely. Never describe the emotional outcome, the subjective result, or what the user will achieve with the product.
4. Never specify a target audience or invent use-cases.
5. Do not infer activities or product categories from the product name. Only use the structured attributes, material, and warranty. For example, do not call a product a "running watch" or "gaming laptop" unless that category is explicitly stated in the attributes.
6. Do not infer environmental values or eco claims from materials. If a material is "recycled" or "sustainably sourced," state that fact directly-do not add "eco-friendly," "minimizing environmental impact," or similar value judgments.
7. NO SUBJECTIVE CLOSURE: Do not conclude the paragraph with a summary of the product’s value or reliability. The description must end on a factual specification.
8. Do not add superlative or ranking claims not in the input (e.g., "industry-leading", "best-in-class", "top-rated", "world's first").
  
**EXAMPLES OF HOW TO TRANSLATE SPECS:**

Input: "200 MP camera"
✓ DO WRITE: "Equipped with a 200 MP camera."
✗ DO NOT WRITE: "Capture life's moments with stunning clarity." (Too emotional/experiential)

Input: "fiber-reinforced polymer"
✓ DO WRITE: "Built with durable fiber-reinforced polymer."
✗ DO NOT WRITE: "Withstands the rigors of your active lifestyle." (Assumes customer lifestyle)

Input: "100% sRGB, Calman Verified, metal stand"
✓ DO WRITE: "Delivers consistent, calibrated color accuracy with 100% sRGB, supported by a metal stand."
✗ DO NOT WRITE: "Perfect for graphic designers and professionals." (Names an audience)

Input: "1080p camera"
✓ DO WRITE: "Equipped with a 1080p camera."
✗ DO NOT WRITE: "Enables high-quality video conferencing." (Invents a use-case for the feature)

**FORMATTING:**
Write one cohesive, flowing paragraph. Vary your sentence structures. Do not use bullet points or lists. Ensure zero spelling or grammatical errors.

**GENERIC RULES**:
1. Be diverse with the vocabulary you use.
2. Do not use your knowledge of the product - only use the attributes provided.

**OUTPUT:**
Return ONLY the product description paragraph. Do not include headers, labels, word counts, or introductory commentary.
"""

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/", 
    api_key=os.environ.get("NEBIUS_API_KEY")                                          
)

# Read CSV into dataframe
df = pd.read_csv("products_dataset.csv")
total = len(df)

results = []                                                                

print(f"Starting inference for {total} products...")
for idx, row in df.head(10).iterrows():
    user_msg = f"""Product: {row['product_name']}
    Attributes: {row['Product_attribute_list']}                                 
    Material: {row['material']}
    Warranty: {row['warranty']}"""                                              
                
    start = time.time()                                                     
    response = client.chat.completions.create(
        model="google/gemma-2-9b-it-fast",
        messages=[                                                          
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}                           
        ],      
        temperature=0.4
    )                                                                       
    latency_ms = (time.time() - start) * 1000
                                                                            
    results.append({
        "product_name": row["product_name"],
        "generated_description": response.choices[0].message.content,       
        "latency_ms": round(latency_ms),
        "input_tokens": response.usage.prompt_tokens,                       
        "output_tokens": response.usage.completion_tokens,
    })

    print(f"[{idx + 1}/{total}] {row['product_name']} - {latency_ms:.0f}ms")

# Calculate average latency
avg_latency_ms = sum(r["latency_ms"] for r in results) / len(results)       
                                                                            
# Calculate average cost per call                                           
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens      
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens     
                                                                            
costs = []                                                                  
for r in results:                                                           
    cost = (r["input_tokens"] * input_price_per_token) + (r["output_tokens"]
* output_price_per_token)                                                  
    costs.append(cost)
                                                                            
avg_cost = sum(costs) / len(costs)

print(f"\nDone! Average latency: {avg_latency_ms:.0f} ms | Average cost: ${avg_cost:.6f}")

# Store results
results_df = pd.DataFrame(results)

for criterion in ["Fluency", "Grammar", "Tone", "Length", "Grounding",      
"Latency", "Cost"]:                                                         
    results_df[criterion] = ""
results_df["final_score"] = ""                                              
                
results_df.to_excel(EXPERIMENT_FILE, index=False)
print(f"Saved to experiment2.xlsx")

# Cost calculation
df_eval = pd.read_excel(EXPERIMENT_FILE)

df_eval["Cost"] = (
    df_eval["input_tokens"] * input_price_per_token
    + df_eval["output_tokens"] * output_price_per_token
)

df_eval.to_excel(EXPERIMENT_FILE, index=False)
df_eval[["product_name", "input_tokens", "output_tokens", "Cost"]]

"""
    What we have changed: 
        - meta-llama/Meta-Llama-3.1-8B-Instruct -> google/gemma-2-9b-it-fast
        - Prompt is completley changed 
        
        
    ┌──────────────┬───────────────────────┬──────────────────────┐                                                                                                         
    │    Metric    │     Experiment 1      │     Experiment 2     │                                                                                                         
    │              │  (Prompt improvement) │  (Gemma 2 9B model)  │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Pass rate    │ 8/10 (80%)            │ 9/10 (90%)           │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Grounding    │ 8 good, 2 ok          │ 10 good              │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Latency      │ All good              │ All good             │                                                                                                         
    │              │ (~1400-2800ms)        │ (~665-1803ms)        │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Cost         │ All good              │ All good             │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Grammar      │ All good              │ All good             │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Fluency      │ 9 good, 1 ok          │ 1 good, 9 ok         │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Tone         │ All good              │ All ok               │                                                                                                         
    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         
    │ Length       │ All good              │ 1 good, 8 ok, 1 bad  │                                                                                                         
    └──────────────┴───────────────────────┴──────────────────────┘                                                                                                         
   
    Discoveries:                                                                                                                                                            
    - Grounding reached 100% good - the strict factual prompt + Gemma 2 9B
      model eliminated all unsupported claims, lifestyle language, and                                                                                                      
      fabricated specs                                                                                                                                                      
    - Trade-off: Tone regressed from all good to all ok - the "strictly                                                                                                     
      factual" prompt made descriptions too dry/clinical (spec-sheet voice)                                                                                                 
    - Fluency dropped from 9 good to 1 good - shorter, formulaic outputs                                                                                                    
      lack sentence variety and cohesion                                                                                                                                    
    - Length is the weakest criterion: most descriptions land at 40-47 words,                                                                                               
      below the 50-word target. iPhone at 37 words is the only Bad rating                                                                                                   
      and the sole cause of failure                                                                                                                                         
    - Latency improved further (~665-1803ms vs ~1400-2800ms)                                                                                                                
    - The model follows grounding rules perfectly but at the cost of warm,                                                                                                  
      natural copy - the next experiment should try to recover Tone and                                                                                                     
      Fluency without regressing Grounding
"""

Starting inference for 50 products...
[11/50] Fitbit Charge 6 — 1215ms
[12/50] GoPro HERO12 Black — 893ms
[13/50] DJI Mini 4 Pro Drone — 730ms
[14/50] Nintendo Switch OLED — 796ms
[15/50] PlayStation 5 Slim — 2336ms
[16/50] Xbox Series X — 911ms
[17/50] Instant Pot Duo 6‑Quart — 1006ms
[18/50] Keurig K‑Supreme Plus Smart — 624ms
[19/50] Vitamix 5200 Blender — 874ms
[20/50] Dyson V15 Detect Vacuum — 1029ms
[21/50] iRobot Roomba j7+ — 847ms
[22/50] Yeti Rambler 20 oz Tumbler — 738ms
[23/50] Stanley Quencher H2.0 40 oz — 730ms
[24/50] Hydro Flask 32 oz Wide Mouth — 954ms
[25/50] Contigo Autoseal West Loop 16 oz — 578ms
[26/50] Nike Air Zoom Pegasus 40 — 1161ms
[27/50] Adidas Ultraboost Light — 738ms
[28/50] The North Face ThermoBall Eco Jacket — 1040ms
[29/50] Patagonia Black Hole 32 L Backpack — 1191ms
[30/50] Osprey Farpoint 40 — 1433ms
[31/50] LEGO Star Wars Millennium Falcon 75192 — 922ms
[32/50] LEGO Icons Botanical Orchid 10311 — 979ms
[33/50] Kindle Paperwhite 11th Gen — 1079ms
[34

'\n    What we have changed: \n        - meta-llama/Meta-Llama-3.1-8B-Instruct -> google/gemma-2-9b-it-fast\n        - Prompt is completley changed \n\n\n    ┌──────────────┬───────────────────────┬──────────────────────┐                                                                                                         \n    │    Metric    │     Experiment 1      │     Experiment 2     │                                                                                                         \n    │              │  (Prompt improvement) │  (Gemma 2 9B model)  │                                                                                                         \n    ├──────────────┼───────────────────────┼──────────────────────┤                                                                                                         \n    │ Pass rate    │ 8/10 (80%)            │ 9/10 (90%)           │                                                                                   

In [ ]:
# Previous Analysis Overall: 9 pass / 1 fail (90%)                                                            
# Biggest pain point: Length - 1 good, 8 ok, 1 bad (10% good)                                                    

# Third (succesful) experiment - Add strict legnth instructions to the prompt
# The previous factual prompt achieved almost perfect grounding but descriptions were consistently
# too short (37-47 words). Since the prompt already forbids filler/marketing fluff, the
# model has no way to pad — so we changed the prompt to allow more filler text.
    
import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

EXPERIMENT_FILE="experiment3.xlsx"

system_prompt="""
You are a technical e-commerce copywriter. Your task is to write a single product description paragraph (exactly 50 to 90 words) based ONLY on the provided product information.

**TONE AND STYLE:**
Write premium, objective catalog copy. Your tone must be friendly, credible grounded entirely in the physical and functional reality of the product. Focus on the engineering, materials, and specifications. Do not use emotional appeals, superlatives, or lifestyle.

**STRICT GROUNDING RULES:**

1. ONLY state the features, materials, and specifications provided in the input.
2. Describe functional capabilities directly tied to the hardware (e.g., "A 30-hour battery provides extended use," or "Bluetooth 5.2 offers wireless connectivity").
3. STOP there. You must leave out the user's experience entirely. Never describe the emotional outcome, the subjective result, or what the user will achieve with the product.
4. Never specify a target audience or invent use-cases.
5. Do not infer activities or product categories from the product name. Only use the structured attributes, material, and warranty. For example, do not call a product a "running watch" or "gaming laptop" unless that category is explicitly stated in the attributes.
6. Do not infer environmental values or eco claims from materials. If a material is "recycled" or "sustainably sourced," state that fact directly-do not add "eco-friendly," "minimizing environmental impact," or similar value judgments.
7. NO SUBJECTIVE CLOSURE: Do not conclude the paragraph with a summary of the product’s value or reliability. The description must end on a factual specification.
8. Do not add superlative or ranking claims not in the input (e.g., "industry-leading", "best-in-class", "top-rated", "world's first").
  
**EXAMPLES OF HOW TO TRANSLATE SPECS:**

Input: "200 MP camera"
✓ DO WRITE: "Equipped with a 200 MP camera."
✗ DO NOT WRITE: "Capture life's moments with stunning clarity." (Too emotional/experiential)

Input: "fiber-reinforced polymer"
✓ DO WRITE: "Built with durable fiber-reinforced polymer."
✗ DO NOT WRITE: "Withstands the rigors of your active lifestyle." (Assumes customer lifestyle)

Input: "100% sRGB, Calman Verified, metal stand"
✓ DO WRITE: "Delivers consistent, calibrated color accuracy with 100% sRGB, supported by a metal stand."
✗ DO NOT WRITE: "Perfect for graphic designers and professionals." (Names an audience)

Input: "1080p camera"
✓ DO WRITE: "Equipped with a 1080p camera."
✗ DO NOT WRITE: "Enables high-quality video conferencing." (Invents a use-case for the feature)

**FORMATTING:**
Write one cohesive, flowing paragraph. Vary your sentence structures. Do not use bullet points or lists. Ensure zero spelling or grammatical errors.

**GENERIC RULES & LENGTH EXPANSION:**
1. Be diverse with the vocabulary you use, but do not use your knowledge of the product - only use the attributes provided.
2. LENGTH REQUIREMENT: You MUST write between 50 and 90 words. Because you are forbidden from using marketing fluff, you must achieve this word count by using detailed, professional sentence structures to describe the facts.
3. HOW TO EXPAND SAFELY: 
   - Spell out technical relationships fully. Instead of "Features an M3 chip," write "The internal architecture is powered by the M3 chip, engineered to deliver optimized computing performance."
   - Use professional transition phrases to connect ideas (e.g., "Furthermore, the device is equipped with...", "In addition to its processing capabilities...", "The hardware configuration also includes...").

**OUTPUT:**
Return ONLY the product description paragraph. Do not include headers, labels, word counts, or introductory commentary.
"""

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/", 
    api_key=os.environ.get("NEBIUS_API_KEY")                                          
)

# Read CSV into dataframe
df = pd.read_csv("products_dataset.csv")
total = len(df)

results = []                                                                

print(f"Starting inference for {total} products...")
for idx, row in df.iterrows():
    user_msg = f"""Product: {row['product_name']}
    Attributes: {row['Product_attribute_list']}                                 
    Material: {row['material']}
    Warranty: {row['warranty']}"""                                              
                
    start = time.time()                                                     
    response = client.chat.completions.create(
        model="google/gemma-2-9b-it-fast",
        messages=[                                                          
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}                           
        ],      
        temperature=0.4
    )                                                                       
    latency_ms = (time.time() - start) * 1000
                                                                            
    results.append({
        "product_name": row["product_name"],
        "generated_description": response.choices[0].message.content,       
        "latency_ms": round(latency_ms),
        "input_tokens": response.usage.prompt_tokens,                       
        "output_tokens": response.usage.completion_tokens,
    })

    print(f"[{idx + 1}/{total}] {row['product_name']} - {latency_ms:.0f}ms")

# Calculate average latency
avg_latency_ms = sum(r["latency_ms"] for r in results) / len(results)       
                                                                            
# Calculate average cost per call                                           
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens      
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens     
                                                                            
costs = []                                                                  
for r in results:                                                           
    cost = (r["input_tokens"] * input_price_per_token) + (r["output_tokens"]
* output_price_per_token)                                                  
    costs.append(cost)
                                                                            
avg_cost = sum(costs) / len(costs)

print(f"\nDone! Average latency: {avg_latency_ms:.0f} ms | Average cost: ${avg_cost:.6f}")

# Store results
results_df = pd.DataFrame(results)

for criterion in ["Fluency", "Grammar", "Tone", "Length", "Grounding",      
"Latency", "Cost"]:                                                         
    results_df[criterion] = ""
results_df["final_score"] = ""                                              
                
results_df.to_excel(EXPERIMENT_FILE, index=False)
print(f"Saved to experiment3.xlsx")

# Cost calculation
df_eval = pd.read_excel(EXPERIMENT_FILE)

df_eval["Cost"] = (
    df_eval["input_tokens"] * input_price_per_token
    + df_eval["output_tokens"] * output_price_per_token
)

df_eval.to_excel(EXPERIMENT_FILE, index=False)
df_eval[["product_name", "input_tokens", "output_tokens", "Cost"]]


# word count
df = pd.read_excel(EXPERIMENT_FILE)
df['Word count'] = df['generated_description'].apply(lambda x: len(str(x).split()))
df.to_excel(EXPERIMENT_FILE, index=False)

"""                                                                
    What we have added:
        - 2. LENGTH REQUIREMENT: You MUST write between 50 and 90 words. Because you are forbidden from using marketing fluff, you must achieve this word count by using detailed, professional sentence structures to describe the facts.
            3. HOW TO EXPAND SAFELY: 
            - Spell out technical relationships fully. Instead of "Features an M3 chip," write "The internal architecture is powered by the M3 chip, engineered to deliver optimized computing performance."
            - Use professional transition phrases to connect ideas (e.g., "Furthermore, the device is equipped with...", "In addition to its processing capabilities...", "The hardware configuration also includes..."). 
                     
    ┌──────────────┬───────────────────────┬──────────────────────┐
    │    Metric    │  Previous Experiment  │  Current Experiment  │
    │              │  (Dry factual prompt) │ (Warm prompt +120tok)│
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Pass rate    │ 9/10 (90%)            │ 6/10 (60%)           │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Grounding    │ 10 good               │ 6 good, 4 ok         │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Latency      │ All good              │ All good             │
    │              │ (~665-1803ms)         │ (~851-2619ms)        │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Cost         │ All good              │ 2 good, 8 ok         │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Grammar      │ All good              │ All good             │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Fluency      │ 1 good, 9 ok          │ All ok               │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Tone         │ All ok                │ All ok               │
    ├──────────────┼───────────────────────┼──────────────────────┤
    │ Length       │ 1 good, 8 ok, 1 bad   │ All good             │
    └──────────────┴───────────────────────┴──────────────────────┘
                                                                    
    Discoveries:                                                   
    - Length is now solved - all 10 products hit 50-90 words (previously only 1 good, 1 bad)
    - Grounding stay the same                 
  """

Starting inference for 50 products...
[11/50] Fitbit Charge 6 — 962ms
[12/50] GoPro HERO12 Black — 852ms
[13/50] DJI Mini 4 Pro Drone — 649ms
[14/50] Nintendo Switch OLED — 1305ms
[15/50] PlayStation 5 Slim — 923ms
[16/50] Xbox Series X — 703ms
[17/50] Instant Pot Duo 6‑Quart — 715ms
[18/50] Keurig K‑Supreme Plus Smart — 819ms
[19/50] Vitamix 5200 Blender — 775ms
[20/50] Dyson V15 Detect Vacuum — 963ms
[21/50] iRobot Roomba j7+ — 800ms
[22/50] Yeti Rambler 20 oz Tumbler — 735ms
[23/50] Stanley Quencher H2.0 40 oz — 1027ms
[24/50] Hydro Flask 32 oz Wide Mouth — 1031ms
[25/50] Contigo Autoseal West Loop 16 oz — 709ms
[26/50] Nike Air Zoom Pegasus 40 — 714ms
[27/50] Adidas Ultraboost Light — 818ms
[28/50] The North Face ThermoBall Eco Jacket — 816ms
[29/50] Patagonia Black Hole 32 L Backpack — 922ms
[30/50] Osprey Farpoint 40 — 819ms
[31/50] LEGO Star Wars Millennium Falcon 75192 — 719ms
[32/50] LEGO Icons Botanical Orchid 10311 — 922ms
[33/50] Kindle Paperwhite 11th Gen — 1319ms
[34/50] 

In [2]:
# new data set 

import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

EXPERIMENT_FILE="experiment4.xlsx"

system_prompt="""
You are a technical e-commerce copywriter. Your task is to write a single product description paragraph (exactly 50 to 90 words) based ONLY on the provided product information.

**TONE AND STYLE:**
Write premium, objective catalog copy. Your tone must be friendly, credible grounded entirely in the physical and functional reality of the product. Focus on the engineering, materials, and specifications. Do not use emotional appeals, superlatives, or lifestyle.

**STRICT GROUNDING RULES:**

1. ONLY state the features, materials, and specifications provided in the input.
2. Describe functional capabilities directly tied to the hardware (e.g., "A 30-hour battery provides extended use," or "Bluetooth 5.2 offers wireless connectivity").
3. STOP there. You must leave out the user's experience entirely. Never describe the emotional outcome, the subjective result, or what the user will achieve with the product.
4. Never specify a target audience or invent use-cases.
5. Do not infer activities or product categories from the product name. Only use the structured attributes, material, and warranty. For example, do not call a product a "running watch" or "gaming laptop" unless that category is explicitly stated in the attributes.
6. Do not infer environmental values or eco claims from materials. If a material is "recycled" or "sustainably sourced," state that fact directly-do not add "eco-friendly," "minimizing environmental impact," or similar value judgments.
7. NO SUBJECTIVE CLOSURE: Do not conclude the paragraph with a summary of the product’s value or reliability. The description must end on a factual specification.
8. Do not add superlative or ranking claims not in the input (e.g., "industry-leading", "best-in-class", "top-rated", "world's first").
  
**EXAMPLES OF HOW TO TRANSLATE SPECS:**

Input: "200 MP camera"
✓ DO WRITE: "Equipped with a 200 MP camera."
✗ DO NOT WRITE: "Capture life's moments with stunning clarity." (Too emotional/experiential)

Input: "fiber-reinforced polymer"
✓ DO WRITE: "Built with durable fiber-reinforced polymer."
✗ DO NOT WRITE: "Withstands the rigors of your active lifestyle." (Assumes customer lifestyle)

Input: "100% sRGB, Calman Verified, metal stand"
✓ DO WRITE: "Delivers consistent, calibrated color accuracy with 100% sRGB, supported by a metal stand."
✗ DO NOT WRITE: "Perfect for graphic designers and professionals." (Names an audience)

Input: "1080p camera"
✓ DO WRITE: "Equipped with a 1080p camera."
✗ DO NOT WRITE: "Enables high-quality video conferencing." (Invents a use-case for the feature)

**FORMATTING:**
Write one cohesive, flowing paragraph. Vary your sentence structures. Do not use bullet points or lists. Ensure zero spelling or grammatical errors.

**GENERIC RULES & LENGTH EXPANSION:**
1. Be diverse with the vocabulary you use, but do not use your knowledge of the product - only use the attributes provided.
2. LENGTH REQUIREMENT: You MUST write between 50 and 90 words. Because you are forbidden from using marketing fluff, you must achieve this word count by using detailed, professional sentence structures to describe the facts.
3. HOW TO EXPAND SAFELY: 
   - Spell out technical relationships fully. Instead of "Features an M3 chip," write "The internal architecture is powered by the M3 chip, engineered to deliver optimized computing performance."
   - Use professional transition phrases to connect ideas (e.g., "Furthermore, the device is equipped with...", "In addition to its processing capabilities...", "The hardware configuration also includes...").

**OUTPUT:**
Return ONLY the product description paragraph. Do not include headers, labels, word counts, or introductory commentary.
"""

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/", 
    api_key=os.environ.get("NEBIUS_API_KEY")                                          
)

# Read CSV into dataframe
df = pd.read_csv("Assignment_01_product_dataset_CHALLENGING.csv")
total = len(df)

results = []                                                                

print(f"Starting inference for {total} products...")
for idx, row in df.iterrows():
    user_msg = f"""Product: {row['product_name']}
    Attributes: {row['Product_attribute_list']}                                 
    Material: {row['material']}
    Warranty: {row['warranty']}"""                                              
                
    start = time.time()                                                     
    response = client.chat.completions.create(
        model="google/gemma-2-9b-it-fast",
        messages=[                                                          
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}                           
        ],      
        temperature=0.4
    )                                                                       
    latency_ms = (time.time() - start) * 1000
                                                                            
    results.append({
        "product_name": row["product_name"],
        "generated_description": response.choices[0].message.content,       
        "latency_ms": round(latency_ms),
        "input_tokens": response.usage.prompt_tokens,                       
        "output_tokens": response.usage.completion_tokens,
    })

    print(f"[{idx + 1}/{total}] {row['product_name']} - {latency_ms:.0f}ms")

# Calculate average latency
avg_latency_ms = sum(r["latency_ms"] for r in results) / len(results)       
                                                                            
# Calculate average cost per call                                           
input_price_per_token = 0.03 / 1_000_000   # $0.03 per 1M input tokens      
output_price_per_token = 0.09 / 1_000_000  # $0.09 per 1M output tokens     
                                                                            
costs = []                                                                  
for r in results:                                                           
    cost = (r["input_tokens"] * input_price_per_token) + (r["output_tokens"]
* output_price_per_token)                                                  
    costs.append(cost)
                                                                            
avg_cost = sum(costs) / len(costs)

print(f"\nDone! Average latency: {avg_latency_ms:.0f} ms | Average cost: ${avg_cost:.6f}")

# Store results
results_df = pd.DataFrame(results)

for criterion in ["Fluency", "Grammar", "Tone", "Length", "Grounding",      
"Latency", "Cost"]:                                                         
    results_df[criterion] = ""
results_df["final_score"] = ""                                              
                
results_df.to_excel(EXPERIMENT_FILE, index=False)
print(f"Saved to experiment3.xlsx")

# Cost calculation
df_eval = pd.read_excel(EXPERIMENT_FILE)

df_eval["Cost"] = (
    df_eval["input_tokens"] * input_price_per_token
    + df_eval["output_tokens"] * output_price_per_token
)

df_eval.to_excel(EXPERIMENT_FILE, index=False)
df_eval[["product_name", "input_tokens", "output_tokens", "Cost"]]


# word count
df = pd.read_excel(EXPERIMENT_FILE)
df['Word count'] = df['generated_description'].apply(lambda x: len(str(x).split()))
df.to_excel(EXPERIMENT_FILE, index=False)



Starting inference for 48 products...
[1/48] EcoBlend Pro Coffee Maker - 1921ms
[2/48] TechNova Wireless Earbuds - 1013ms
[3/48] FitTrack Smart Scale - 741ms
[4/48] UltraGrip Kitchen Knife Set - 1003ms
[5/48] AquaPure Water Filter Pitcher - 1692ms
[6/48] CloudRest Memory Foam Pillow - 1210ms
[7/48] PowerMax Portable Charger - 771ms
[8/48] HomeGuard Security Camera - 1032ms
[9/48] SpeedDry Hair Dryer - 864ms
[10/48] FlexiBoard Cutting Board Set - 978ms
[11/48] TurboVac Handheld Vacuum - 823ms
[12/48] BrightView Desk Lamp - 691ms
[13/48] ChillBox Mini Fridge - 760ms
[14/48] ComfortSeat Office Chair - 919ms
[15/48] PureAir Humidifier - 1216ms
[16/48] SonicClean Electric Toothbrush - 1024ms
[17/48] QuickSnap Smartphone Mount - 914ms
[18/48] NutriBlend Personal Blender - 1042ms
[19/48] RestEasy Mattress Topper - 901ms
[20/48] ProClip Nail Clippers - 1474ms
[21/48] SafeGrip Shower Mat - 800ms
[22/48] ClearView Screen Protector - 1276ms
[23/48] SoftStep Yoga Mat - 1193ms
[24/48] CompactFold L

## Task 5 - create a judge model (20 points):
In the previous tasks you manually evaluated each generated description - thorough but
slow and hard to scale. In this task you'll build an automated judge: an LLM that grades
descriptions using the same rubric you defined in Task 1.
For this you’ll need to choose the model, write a judge prompt and define the output schema.

### Model
 Start with the model you did not use in Task 2. If you find it struggles as a judge, you may switch to a larger model (e.g., Qwen3-30B-A3B-Instruct-2507). Document why you made the switch.
### Prompt
 Write a judge prompt that includes your rubric definitions so the model applies the
same standards you used during manual evaluation. Exclude cost and latency criteria - those are measured programmatically. Think carefully about what context the judge needs to evaluate each criterion - especially Grounding.

### Output
for each criterion the judge should return:
- explanation (string) — reasoning for the verdict
- verdict (enum: good / ok / bad)

Note that explanation comes before verdict in the schema. In your submission, explain why
this ordering matters.
Use a Pydantic schema to enforce this structure via the API's structured output support.
Here you’ll find a short explanation as to why using Pydantic.

### Why explanation comes first
The explanation comes before the verdict because putting the explanation first, you force the model into a chain-of-thought. If you ask for the verdict first, the model will just commit to a label and hallucinate a justification for it.

In [2]:
import os
import time

import pandas as pd
from dotenv import load_dotenv
from enum import Enum
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()

class Verdict(str, Enum):
    good = "good"
    ok = "ok"
    bad = "bad"


class CriterionResult(BaseModel):
    explanation: str
    verdict: Verdict


class JudgeOutput(BaseModel):
    grounding: CriterionResult
    grammar: CriterionResult
    fluency: CriterionResult
    tone: CriterionResult

JUDGE_SYSTEM_PROMPT = """\
You are a strict, impartial quality-assurance judge for e-commerce product descriptions.

You will receive:
- The ORIGINAL PRODUCT DATA (name, attributes, material, warranty).
- The GENERATED DESCRIPTION to evaluate.

For each criterion below, return an `explanation` (your reasoning) and a `verdict` (good / ok / bad).

---

### Grounding — "Sticks to information provided"
Evaluate whether factual claims are supported by the provided product attributes.
Grounding does NOT evaluate writing quality (Fluency), voice (Tone), or mechanical correctness (Grammar).

* **good**: Every factual claim is directly supported by the input. Generic marketing language ("great choice", "you'll love it") is acceptable. The following inferences ARE allowed:
  - Physical/sensory properties from materials (steel → durable, cotton → soft, leather → premium-feeling)
  - Functional capabilities from attributes (Bluetooth 5.0 → wireless connectivity, 12-speed gear → versatile shifting)
* **ok**: Mostly faithful, but contains AT LEAST one unsupported-but-plausible opinion or suggestion — a subjective claim not in the input that doesn't contradict it (e.g., "perfect for hiking", "makes a great gift", "eco-friendly choice", "ideal for commuters").
* **bad**: Contradicts provided features, OR contains fabricated specific/technical claims not in the input (e.g., "waterproof to 30 meters", "available in 5 colors", "FDA-approved").

The key distinction between ok and bad: if the unsupported claim is a subjective opinion that cannot be verified as true or false, it is ok. If the unsupported claim is a specific, verifiable fact that could be checked and confirmed or denied, it is bad.

### Grammar — correct spelling & punctuation
Grammar evaluates mechanical correctness only. It does NOT evaluate whether the text flows well (Fluency) or uses the right voice (Tone).

* **good**: No spelling, punctuation, or grammatical errors.
* **ok**: 1-2 minor errors that don't hinder readability.
* **bad**: 3+ errors, OR any error that changes meaning or makes a sentence hard to understand.

### Fluency — "Natural, easy-to-read sentences"
Fluency evaluates whether the description reads smoothly as coherent, human-sounding prose. It does NOT evaluate whether the voice is appropriate for e-commerce (Tone), whether the facts are accurate (Grounding), or whether spelling and punctuation are correct (Grammar).

Two components:
- **Flow/cohesion**: reads as a unified paragraph, not disconnected facts. Sentences reference or build on each other, related features grouped, logical progression.
- **Naturalness**: sounds like a human copywriter. Varied sentence structure, no awkward phrasing, doesn't read like bullet points reformatted into sentences.

* **good**: Both satisfied — cohesive flow and natural-sounding.
* **ok**: One breaks down (e.g., natural phrasing but disconnected, or well-structured but robotic/repetitive).
* **bad**: Both fail — disjointed facts in awkward or repetitive phrasing.

### Tone — "Friendly, credible sales voice"
Tone evaluates whether the description uses the right voice for e-commerce product copy. It does NOT evaluate whether the text flows well (Fluency) or is mechanically correct (Grammar).

Two components:
- **Friendliness**: warm, approachable, benefit-oriented. Speaks TO the customer.
- **Credibility**: confident but measured. No unsupported superlatives, no high-pressure urgency, no stacked vague intensifiers.

* **good**: Both satisfied — warm and inviting without overselling.
* **ok**: Leans too far one way — too dry/clinical OR mildly over-enthusiastic but still broadly appropriate.
* **bad**: Clearly wrong voice — reads like a technical manual with zero sales appeal, OR aggressively pushy/hyperbolic.
"""

JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Gemma 2 9B pricing on Nebius (generation model)
GEN_INPUT_PRICE  = 0.03 / 1_000_000   # $/token
GEN_OUTPUT_PRICE = 0.09 / 1_000_000

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)


def judge_length(text: str) -> str:
    """Rule-based length verdict."""
    wc = len(text.split())
    if 50 <= wc <= 90:
        return "good"
    elif 40 <= wc <= 110:
        return "ok"
    return "bad"


def judge_cost(cost: float) -> str:
    """Rule-based cost verdict (per-call generation cost)."""
    if cost <= 0.00006:
        return "good"
    elif cost <= 0.0001:
        return "ok"
    return "bad"


def judge_latency(latency_ms: float) -> str:
    """Rule-based latency verdict (per-call generation latency)."""
    if latency_ms <= 5000:
        return "good"
    elif latency_ms <= 8000:
        return "ok"
    return "bad"


def build_user_prompt(product_row, description: str) -> str:
    return (
        "## Original Product Data\n"
        f"Product: {product_row['product_name']}\n"
        f"Attributes: {product_row['Product_attribute_list']}\n"
        f"Material: {product_row['material']}\n"
        f"Warranty: {product_row['warranty']}\n\n"
        "## Generated Description to Evaluate\n"
        f"{description}"
    )

In [ ]:
eval_df = pd.read_excel("experiment4.xlsx")
products_df = pd.read_csv("Assignment_01_product_dataset_CHALLENGING.csv")

judge_results = []
print(f"Judging {len(eval_df)} descriptions with {JUDGE_MODEL}...")

for idx, row in eval_df.iterrows():
    product_row = products_df[products_df["product_name"] == row["product_name"]].iloc[0]
    user_prompt = build_user_prompt(product_row, row["generated_description"])

    start = time.time()
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        extra_body={"guided_json": JudgeOutput.model_json_schema()},
    )
    latency_ms = (time.time() - start) * 1000

    parsed = JudgeOutput.model_validate_json(response.choices[0].message.content)
    word_count = len(row["generated_description"].split())

    # Generation cost & latency per call
    gen_cost = row["input_tokens"] * GEN_INPUT_PRICE + row["output_tokens"] * GEN_OUTPUT_PRICE
    gen_latency_ms = row["latency_ms"]

    judge_results.append({
        "product_name":          row["product_name"],
        "grounding_verdict":     parsed.grounding.verdict.value,
        "grounding_explanation": parsed.grounding.explanation,
        "length_verdict":        judge_length(row["generated_description"]),
        "length_explanation":    f"Word count: {word_count}",
        "cost_verdict":          judge_cost(gen_cost),
        "cost_explanation":      f"${gen_cost:.6f} per call",
        "latency_verdict":       judge_latency(gen_latency_ms),
        "latency_explanation":   f"{gen_latency_ms} ms",
        "grammar_verdict":       parsed.grammar.verdict.value,
        "grammar_explanation":   parsed.grammar.explanation,
        "fluency_verdict":       parsed.fluency.verdict.value,
        "fluency_explanation":   parsed.fluency.explanation,
        "tone_verdict":          parsed.tone.verdict.value,
        "tone_explanation":      parsed.tone.explanation,
        "judge_latency_ms":      round(latency_ms),
        "judge_input_tokens":    response.usage.prompt_tokens,
        "judge_output_tokens":   response.usage.completion_tokens,
    })
    print(f"  [{idx + 1}/{len(eval_df)}] {row['product_name']} — {latency_ms:.0f} ms")

judge_df = pd.DataFrame(judge_results)

# Llama 3.1 8B pricing on Nebius (judge model)
JUDGE_INPUT_PRICE  = 0.03 / 1_000_000   # $/token
JUDGE_OUTPUT_PRICE = 0.09 / 1_000_000

judge_df["judge_cost"] = (
    judge_df["judge_input_tokens"] * JUDGE_INPUT_PRICE
    + judge_df["judge_output_tokens"] * JUDGE_OUTPUT_PRICE
)

print("--- Judge Stats ---")
print(f"Avg latency:  {judge_df['judge_latency_ms'].mean():.0f} ms")
print(f"Avg cost:     ${judge_df['judge_cost'].mean():.6f}")
print(f"Total cost:   ${judge_df['judge_cost'].sum():.6f}")

print("\n--- Verdicts ---")
verdict_cols = ["product_name", "grounding_verdict", "length_verdict",
                "cost_verdict", "latency_verdict",
                "grammar_verdict", "fluency_verdict", "tone_verdict"]
print(judge_df[verdict_cols].to_string(index=False))

print("\n--- Summary ---")
for criterion in ["grounding", "length", "cost", "latency", "grammar", "fluency", "tone"]:
    counts = judge_df[f"{criterion}_verdict"].value_counts()
    print(f"{criterion.capitalize():12s}: {counts.to_dict()}")

judge_df

## Task 6 - run and analyze the judge:
1. Sanity check - run the judge on 5 products. Review the explanations and verdicts
manually — do they make sense? Does the judge apply your rubric correctly? Adjust
the prompt if needed before proceeding.
2. Full run - run the judge on all products. Compute the final_score for each product
using the same pass bar and go/no-go rules from Task 1. Store everything back into
the spreadsheet.
3. Compare to human evaluation - for each criterion, compare the judge's verdicts
against your manual scores from Task 3. Compute an agreement rate per criterion.
Where do they agree? Where do they diverge? Try to explain why.
4. Criterion-by-criterion judging - instead of asking the judge to evaluate all criteria at
once, run it separately for each criterion (one call per criterion per product). Does this
change the results? Why might isolating criteria lead to different outcomes? Compare
agreement with your human scores — did it improve?
5. Analysis - reflect on what you found:
a. What are the practical trade-offs between human evaluation and
LLM-as-a-judge (think: cost, scale, consistency, accuracy)?
b. Which approach would you recommend for a production system that
generates thousands of descriptions daily?

In [5]:
# 6.1 - Sanity check: run judge on 5 products
# The results actually don't make sense because the LLM's classifications are too lenient
# the judge is giving much more "good" than we did.

import time
import pandas as pd

SANITY_N = 5

sanity_eval_df = pd.read_excel("experiment4.xlsx").head(SANITY_N)
sanity_products_df = pd.read_csv("Assignment_01_product_dataset_CHALLENGING.csv")


def compute_pass(verdicts: dict) -> tuple:
    """Apply Task 1 pass bar and go/no-go rules."""
    all_v = list(verdicts.values())
    good_count = all_v.count("good")
    bad_count = all_v.count("bad")
    passed = True
    reasons = []

    if verdicts["grounding"] != "good":
        reasons.append(f"Grounding is {verdicts['grounding']} (must be good)")
        passed = False
    if bad_count > 0:
        bad_names = [k for k, v in verdicts.items() if v == "bad"]
        reasons.append(f"{bad_count} bad verdict(s): {', '.join(bad_names)}")
        passed = False
    if good_count < 3:
        reasons.append(f"Only {good_count} good (need >=3)")
        passed = False
    gft = [verdicts["grammar"], verdicts["fluency"], verdicts["tone"]]
    if all(v == "ok" for v in gft):
        reasons.append("Grammar+Fluency+Tone all OK")
        passed = False

    return passed, "; ".join(reasons) if reasons else "All checks passed"


sanity_results = []
print(f"Running judge on {SANITY_N} products for sanity check...\n")

for idx, row in sanity_eval_df.iterrows():
    product_row = sanity_products_df[
        sanity_products_df["product_name"] == row["product_name"]
    ].iloc[0]
    user_prompt = build_user_prompt(product_row, row["generated_description"])

    start = time.time()
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        extra_body={"guided_json": JudgeOutput.model_json_schema()},
    )
    latency_ms = (time.time() - start) * 1000
    parsed = JudgeOutput.model_validate_json(response.choices[0].message.content)

    # Programmatic verdicts
    gen_cost = row["input_tokens"] * GEN_INPUT_PRICE + row["output_tokens"] * GEN_OUTPUT_PRICE
    length_v = judge_length(row["generated_description"])
    cost_v = judge_cost(gen_cost)
    latency_v = judge_latency(row["latency_ms"])
    wc = len(row["generated_description"].split())

    verdicts = {
        "grounding": parsed.grounding.verdict.value,
        "grammar":   parsed.grammar.verdict.value,
        "fluency":   parsed.fluency.verdict.value,
        "tone":      parsed.tone.verdict.value,
        "length":    length_v,
        "cost":      cost_v,
        "latency":   latency_v,
    }
    passed, reason = compute_pass(verdicts)

    sanity_results.append({
        "product_name": row["product_name"],
        "verdicts": verdicts,
        "parsed": parsed,
        "gen_cost": gen_cost,
        "wc": wc,
        "passed": passed,
        "reason": reason,
        "product_row": product_row,
        "description": row["generated_description"],
        "latency_ms": row["latency_ms"],
    })
    print(f"  [{idx + 1}/{SANITY_N}] {row['product_name']} — {latency_ms:.0f} ms")

# ── Display results ──
print("\n" + "=" * 80)
for i, r in enumerate(sanity_results):
    p = r["product_row"]
    v = r["verdicts"]
    pr = r["parsed"]

    print(f"\n{'━' * 70}")
    print(f"  Product {i+1}/{SANITY_N}: {r['product_name']}")
    print(f"{'━' * 70}")

    print(f"\n  ORIGINAL DATA:")
    print(f"    Attributes: {p['Product_attribute_list']}")
    print(f"    Material:   {p['material']}")
    print(f"    Warranty:   {p['warranty']}")

    print(f"\n  GENERATED DESCRIPTION ({r['wc']} words):")
    print(f"    \"{r['description']}\"")

    print(f"\n  VERDICTS:")
    print(f"    Grounding:  {v['grounding'].upper():4s}  — {pr.grounding.explanation}")
    print(f"    Grammar:    {v['grammar'].upper():4s}  — {pr.grammar.explanation}")
    print(f"    Fluency:    {v['fluency'].upper():4s}  — {pr.fluency.explanation}")
    print(f"    Tone:       {v['tone'].upper():4s}  — {pr.tone.explanation}")
    print(f"    Length:     {v['length'].upper():4s}  — Word count: {r['wc']}")
    print(f"    Cost:       {v['cost'].upper():4s}  — ${r['gen_cost']:.6f} per call")
    print(f"    Latency:    {v['latency'].upper():4s}  — {r['latency_ms']} ms")

    good_n = list(v.values()).count("good")
    ok_n = list(v.values()).count("ok")
    bad_n = list(v.values()).count("bad")
    status = "PASS" if r["passed"] else "FAIL"
    print(f"\n  PASS BAR: good={good_n} | ok={ok_n} | bad={bad_n}")
    print(f"  → {status}  ({r['reason']})")

print(f"\n{'━' * 70}")
print("Sanity check complete. Review the verdicts and explanations above.")

Running judge on 5 products for sanity check...

  [1/5] EcoBlend Pro Coffee Maker — 4524 ms
  [2/5] TechNova Wireless Earbuds — 4425 ms
  [3/5] FitTrack Smart Scale — 3718 ms
  [4/5] UltraGrip Kitchen Knife Set — 3603 ms
  [5/5] AquaPure Water Filter Pitcher — 12058 ms


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Product 1/5: EcoBlend Pro Coffee Maker
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ORIGINAL DATA:
    Attributes: features: 12-cup capacity, programmable timer, auto-shutoff after 2 hours; note: NOT dishwasher safe, manual cleaning only
    Material:   glass carafe, BPA-free plastic housing
    Warranty:   90-day limited warranty

  GENERATED DESCRIPTION (71 words):
    "The EcoBlend Pro Coffee Maker features a 12-cup capacity, allowing for larger coffee batches.  Programmable timer functionality enables users to schedule brewing in advance.  For safety, the device automatically shuts off after 2 hours of inactivity.

In [7]:
# 6.2 - Full run

import time
import pandas as pd

SANITY_N = 48

sanity_eval_df = pd.read_excel("experiment4.xlsx").head(SANITY_N)
sanity_products_df = pd.read_csv("Assignment_01_product_dataset_CHALLENGING.csv")


def compute_pass(verdicts: dict) -> tuple:
    """Apply Task 1 pass bar and go/no-go rules."""
    all_v = list(verdicts.values())
    good_count = all_v.count("good")
    bad_count = all_v.count("bad")
    passed = True
    reasons = []

    if verdicts["grounding"] != "good":
        reasons.append(f"Grounding is {verdicts['grounding']} (must be good)")
        passed = False
    if bad_count > 0:
        bad_names = [k for k, v in verdicts.items() if v == "bad"]
        reasons.append(f"{bad_count} bad verdict(s): {', '.join(bad_names)}")
        passed = False
    if good_count < 3:
        reasons.append(f"Only {good_count} good (need >=3)")
        passed = False
    gft = [verdicts["grammar"], verdicts["fluency"], verdicts["tone"]]
    if all(v == "ok" for v in gft):
        reasons.append("Grammar+Fluency+Tone all OK")
        passed = False

    return passed, "; ".join(reasons) if reasons else "All checks passed"


sanity_results = []
print(f"Running judge on {SANITY_N} products for sanity check...\n")

for idx, row in sanity_eval_df.iterrows():
    product_row = sanity_products_df[
        sanity_products_df["product_name"] == row["product_name"]
    ].iloc[0]
    user_prompt = build_user_prompt(product_row, row["generated_description"])

    start = time.time()
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        extra_body={"guided_json": JudgeOutput.model_json_schema()},
    )
    latency_ms = (time.time() - start) * 1000
    parsed = JudgeOutput.model_validate_json(response.choices[0].message.content)

    # Programmatic verdicts
    gen_cost = row["input_tokens"] * GEN_INPUT_PRICE + row["output_tokens"] * GEN_OUTPUT_PRICE
    length_v = judge_length(row["generated_description"])
    cost_v = judge_cost(gen_cost)
    latency_v = judge_latency(row["latency_ms"])
    wc = len(row["generated_description"].split())

    verdicts = {
        "grounding": parsed.grounding.verdict.value,
        "grammar":   parsed.grammar.verdict.value,
        "fluency":   parsed.fluency.verdict.value,
        "tone":      parsed.tone.verdict.value,
        "length":    length_v,
        "cost":      cost_v,
        "latency":   latency_v,
    }
    passed, reason = compute_pass(verdicts)

    sanity_results.append({
        "product_name": row["product_name"],
        "verdicts": verdicts,
        "parsed": parsed,
        "gen_cost": gen_cost,
        "wc": wc,
        "passed": passed,
        "reason": reason,
        "product_row": product_row,
        "description": row["generated_description"],
        "latency_ms": row["latency_ms"],
    })
    print(f"  [{idx + 1}/{SANITY_N}] {row['product_name']} — {latency_ms:.0f} ms")

# Store results
rows = []
for r in sanity_results:
    v = r["verdicts"]
    rows.append({
        "product_name": r["product_name"],
        "Grounding": v["grounding"].capitalize(),
        "Grammar": v["grammar"].capitalize(),
        "Fluency": v["fluency"].capitalize(),
        "Tone": v["tone"].capitalize(),
        "Length": v["length"].capitalize(),
        "Latency": v["latency"].capitalize(),
        "Cost": v["cost"].capitalize(),
        "final_score": "Pass" if r["passed"] else "Fail",
    })

task6_df = pd.DataFrame(rows)
task6_df.to_excel("assignment_01_task6.xlsx", index=False)
print(f"Wrote {len(task6_df)} rows to assignment_01_task6.xlsx")
print(f"Pass/Fail breakdown:")
print(task6_df["final_score"].value_counts().to_string())
print(f"{task6_df.head(10).to_string()}")

# ── Display results ──
print("\n" + "=" * 80)
for i, r in enumerate(sanity_results):
    p = r["product_row"]
    v = r["verdicts"]
    pr = r["parsed"]

    print(f"\n{'━' * 70}")
    print(f"  Product {i+1}/{SANITY_N}: {r['product_name']}")
    print(f"{'━' * 70}")

    print(f"\n  ORIGINAL DATA:")
    print(f"    Attributes: {p['Product_attribute_list']}")
    print(f"    Material:   {p['material']}")
    print(f"    Warranty:   {p['warranty']}")

    print(f"\n  GENERATED DESCRIPTION ({r['wc']} words):")
    print(f"    \"{r['description']}\"")

    print(f"\n  VERDICTS:")
    print(f"    Grounding:  {v['grounding'].upper():4s}  — {pr.grounding.explanation}")
    print(f"    Grammar:    {v['grammar'].upper():4s}  — {pr.grammar.explanation}")
    print(f"    Fluency:    {v['fluency'].upper():4s}  — {pr.fluency.explanation}")
    print(f"    Tone:       {v['tone'].upper():4s}  — {pr.tone.explanation}")
    print(f"    Length:     {v['length'].upper():4s}  — Word count: {r['wc']}")
    print(f"    Cost:       {v['cost'].upper():4s}  — ${r['gen_cost']:.6f} per call")
    print(f"    Latency:    {v['latency'].upper():4s}  — {r['latency_ms']} ms")

    good_n = list(v.values()).count("good")
    ok_n = list(v.values()).count("ok")
    bad_n = list(v.values()).count("bad")
    status = "PASS" if r["passed"] else "FAIL"
    print(f"\n  PASS BAR: good={good_n} | ok={ok_n} | bad={bad_n}")
    print(f"  → {status}  ({r['reason']})")

print(f"\n{'━' * 70}")
print("Sanity check complete. Review the verdicts and explanations above.")

Running judge on 48 products for sanity check...

  [1/48] EcoBlend Pro Coffee Maker — 11729 ms
  [2/48] TechNova Wireless Earbuds — 9020 ms
  [3/48] FitTrack Smart Scale — 7318 ms
  [4/48] UltraGrip Kitchen Knife Set — 7082 ms
  [5/48] AquaPure Water Filter Pitcher — 6346 ms
  [6/48] CloudRest Memory Foam Pillow — 5140 ms
  [7/48] PowerMax Portable Charger — 6337 ms
  [8/48] HomeGuard Security Camera — 6044 ms
  [9/48] SpeedDry Hair Dryer — 6741 ms
  [10/48] FlexiBoard Cutting Board Set — 5470 ms
  [11/48] TurboVac Handheld Vacuum — 5320 ms
  [12/48] BrightView Desk Lamp — 9774 ms
  [13/48] ChillBox Mini Fridge — 7692 ms
  [14/48] ComfortSeat Office Chair — 6063 ms
  [15/48] PureAir Humidifier — 6633 ms
  [16/48] SonicClean Electric Toothbrush — 5795 ms
  [17/48] QuickSnap Smartphone Mount — 6698 ms
  [18/48] NutriBlend Personal Blender — 8802 ms
  [19/48] RestEasy Mattress Topper — 5752 ms
  [20/48] ProClip Nail Clippers — 7458 ms
  [21/48] SafeGrip Shower Mat — 6962 ms
  [22/48] Cle

In [8]:
# 6.4 - Agreement rate: human (Task 3) vs judge (Task 6)
# its task 4 actually because its after all the improvement cycles

import pandas as pd

N_ROWS = 10  # the first ten rows because that's what we evaluate manually

human_df = pd.read_excel("assignment_01_task4.xlsx").head(N_ROWS)
judge_df = pd.read_excel("assignment_01_task6.xlsx").head(N_ROWS)

# Align by product_name to be safe
merged = human_df.merge(judge_df, on="product_name", suffixes=("_human", "_judge"))

criteria = ["Grounding", "Grammar", "Fluency", "Tone", "Length", "Latency", "Cost"]

print(f"Comparing first {N_ROWS} products")
print(f"{"Criterion":<12} {"Agree":>5} / {"Total":>5}   {"Agreement Rate":>15}")
print("-" * 45)

total_agree = 0
total_comparisons = 0

for criterion in criteria:
    h = merged[f"{criterion}_human"].str.strip().str.lower()
    j = merged[f"{criterion}_judge"].str.strip().str.lower()
    agree = (h == j).sum()
    total = len(merged)
    rate = agree / total * 100
    total_agree += agree
    total_comparisons += total
    print(f"{criterion:<12} {agree:>5} / {total:>5}   {rate:>14.1f}%")

overall = total_agree / total_comparisons * 100
print("-" * 45)
print(f"{"Overall":<12} {total_agree:>5} / {total_comparisons:>5}   {overall:>14.1f}%")

# Show disagreements
print(f"Disagreements:")
for criterion in criteria:
    h = merged[f"{criterion}_human"].str.strip().str.lower()
    j = merged[f"{criterion}_judge"].str.strip().str.lower()
    mask = h != j
    if mask.any():
        print(f"  {criterion}:")
        for _, row in merged[mask].iterrows():
            print(f"    {row["product_name"]}: human={row[f"{criterion}_human"]} vs judge={row[f"{criterion}_judge"]}")
        print()


Comparing first 10 products
Criterion    Agree / Total    Agreement Rate
---------------------------------------------
Grounding       10 /    10            100.0%
Grammar         10 /    10            100.0%
Fluency          3 /    10             30.0%
Tone             3 /    10             30.0%
Length          10 /    10            100.0%
Latency         10 /    10            100.0%
Cost            10 /    10            100.0%
---------------------------------------------
Overall         56 /    70             80.0%
Disagreements:
  Fluency:
    EcoBlend Pro Coffee Maker: human=Ok vs judge=Good
    FitTrack Smart Scale: human=Ok vs judge=Good
    UltraGrip Kitchen Knife Set: human=Ok vs judge=Good
    AquaPure Water Filter Pitcher: human=Ok vs judge=Good
    CloudRest Memory Foam Pillow: human=Ok vs judge=Good
    HomeGuard Security Camera: human=Ok vs judge=Good
    FlexiBoard Cutting Board Set: human=Ok vs judge=Good

  Tone:
    EcoBlend Pro Coffee Maker: human=Ok vs judge=Good
 

By the results we cleary can see that the Fluecy and Tone are the 2 most inaccurate evaluations that the judge did. as we stated in 6.1 the judge is giving much more "good" than we did. our hypothosis is that because the LLM tends to be more "helpful assistant" in general, it classify more "good"s than a human evaluator would do. especially, in a criteria as general and personal as "Tone" and "Fluency"